# 06 Export Main Embeddings

Export CLS and mean pooled embeddings from a checkpoint. Change `checkpoint_path` in `configs/export_embeddings.yaml` to re-export from another checkpoint.

In [1]:
from pathlib import Path
import copy
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/root/projects/bert4rec')

In [2]:
from src.export_embeddings import export_main_embeddings, load_export_embeddings_config

base_config = load_export_embeddings_config(PROJECT_ROOT / "configs" / "export_embeddings.yaml")
base_config

ExportEmbeddingsConfig(checkpoint_path='artifacts/checkpoints/checkpoint_best.pt', event_vocab_path='artifacts/vocab/event_token_vocab.json', train_config_path='configs/train.yaml', manifest_path='artifacts/manifests/main_embeddings_manifest.json', input_prefixes={'train': 'data/processed/train_prefixes.parquet', 'valid': 'data/processed/valid_prefixes.parquet', 'test': 'data/processed/test_prefixes.parquet'}, outputs={'cls': 'data/exports/main_embeddings_cls.parquet', 'mean': 'data/exports/main_embeddings_mean.parquet'}, batch_size=512, num_workers=0, device='auto', mixed_precision=True, dry_run_rows=None)

In [3]:
DRY_RUN = False
config = copy.deepcopy(base_config)
if DRY_RUN:
    config = config.__class__(**{**config.__dict__, "dry_run_rows": 256})

print("DRY_RUN", DRY_RUN)
print("checkpoint", config.checkpoint_path)
print("dry_run_rows", config.dry_run_rows)

DRY_RUN False
checkpoint artifacts/checkpoints/checkpoint_best.pt
dry_run_rows None


In [4]:
manifest = export_main_embeddings(config, project_root=PROJECT_ROOT)
manifest

/root/projects/bert4rec/src/model_event_encoder.py:48: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.n_layers)


Export train embeddings:   0%|          | 0/190 [00:00<?, ?it/s]

Export valid embeddings:   0%|          | 0/42 [00:00<?, ?it/s]

Export test embeddings:   0%|          | 0/41 [00:00<?, ?it/s]

{'created_at': '2026-05-10T13:49:29.758184+00:00',
 'checkpoint_path': 'artifacts/checkpoints/checkpoint_best.pt',
 'event_vocab_path': 'artifacts/vocab/event_token_vocab.json',
 'train_config_path': 'configs/train.yaml',
 'device': 'cuda',
 'dry_run_rows': None,
 'split_counts': {'train': 97122, 'valid': 21148, 'test': 20853},
 'outputs': {'cls': 'data/exports/main_embeddings_cls.parquet',
  'mean': 'data/exports/main_embeddings_mean.parquet'}}

In [5]:
import pandas as pd
cls = pd.read_parquet(PROJECT_ROOT / config.outputs["cls"])
mean = pd.read_parquet(PROJECT_ROOT / config.outputs["mean"])
print(cls.shape, mean.shape)
cls.head()

(139123, 14) (139123, 14)


,user_id,split,prefix_len,num_events_total,prefix_start_ts,prefix_end_ts,next_event_token_id,label_available_retention_7d,label_retention_7d,label_available_retention_14d,label_retention_14d,embedding,pooling,checkpoint_path
0,10000235382394935150,train,50,75,1773673777,1773673783,62.0,True,0.0,False,NaN,"[0.14254236, 11.258975, 20.374237, -21.793756,...",cls,artifacts/checkpoints/checkpoint_best.pt
1,10000279983408954705,train,50,406,1774705665,1774705676,407.0,False,NaN,False,NaN,"[-5.4445243, 10.221181, -21.879425, 7.150824, ...",cls,artifacts/checkpoints/checkpoint_best.pt
2,10000279983408954705,train,100,406,1774705665,1774705707,77.0,False,NaN,False,NaN,"[6.976267, 7.000247, 12.776337, -1.3619933, 3....",cls,artifacts/checkpoints/checkpoint_best.pt
3,10000279983408954705,train,150,406,1774705665,1774714019,74.0,False,NaN,False,NaN,"[-0.18796301, 11.242412, 3.5462344, 0.09357309...",cls,artifacts/checkpoints/checkpoint_best.pt
4,10000413951987079100,train,50,707,1772150043,1772150050,60.0,True,0.0,True,0.0,"[6.8634286, 15.607974, -2.6692197, -7.7285705,...",cls,artifacts/checkpoints/checkpoint_best.pt
